In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import warnings
warnings.filterwarnings('ignore')

random_seed = 123
sns.set_style('whitegrid')
print('All libraries imported successfully')

In [ ]:
# Load and process data
df = pd.read_csv('datafile_full.csv')
print(f'Original data shape: {df.shape}')
print(f'Target distribution:')
print(df['target'].value_counts())

# Handle missing values
missing_pct = (df.isnull().sum() / len(df)) * 100
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()
print(f'\nDropping {len(cols_to_drop)} columns with >40% missing values')
df = df.drop(columns=cols_to_drop)

feature_cols = [col for col in df.columns if col.startswith('var_')]
imputer = SimpleImputer(strategy='mean')
df[feature_cols] = imputer.fit_transform(df[feature_cols])

print(f'Shape after missing value handling: {df.shape}')
print(f'Missing values remaining: {df.isnull().sum().sum()}')

In [ ]:
# Initial train-test split (before feature selection)
X_full = df[[col for col in df.columns if col.startswith('var_')]]
y_full = df['target']

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.3, random_state=random_seed, stratify=y_full
)

print(f'Training set size: {X_train_full.shape}')
print(f'Test set size: {X_test_full.shape}')
print(f'Total features available: {X_train_full.shape[1]}')

In [ ]:
# Define feature selection function with IG threshold
def select_features_by_threshold(X_train, y_train, X_test, threshold):
    """
    Select features based on Information Gain (f_classif) threshold
    
    Parameters:
    -----------
    X_train : array-like
        Training features
    y_train : array-like
        Training target
    X_test : array-like
        Test features
    threshold : float
        Ratio of top features to keep (0.0-1.0)
        e.g., 0.7 means keep top 70% of features
    
    Returns:
    --------
    X_train_selected : DataFrame
        Selected training features
    X_test_selected : DataFrame
        Selected test features
    selected_features : list
        Names of selected features
    scores : array
        Feature importance scores
    """
    total_features = X_train.shape[1]
    
    if threshold >= 1.0:
        # Keep all features
        return X_train, X_test, list(X_train.columns), None
    
    # Calculate k (number of features to keep)
    k = max(1, int(total_features * threshold))
    
    # Feature selection using Information Gain
    selector = SelectKBest(f_classif, k=k)
    X_train_selected_array = selector.fit_transform(X_train, y_train)
    X_test_selected_array = selector.transform(X_test)
    
    # Get selected feature indices
    selected_indices = selector.get_support(indices=True)
    selected_features = [X_train.columns[i] for i in selected_indices]
    
    # Create DataFrames with selected features
    X_train_selected = pd.DataFrame(X_train_selected_array, columns=selected_features)
    X_test_selected = pd.DataFrame(X_test_selected_array, columns=selected_features)
    
    return X_train_selected, X_test_selected, selected_features, selector.scores_

print('Feature selection function defined successfully')

In [ ]:
# Define model evaluation function
def evaluate_lda_model(X_train, y_train, X_test, y_test, model_name):
    """
    Train and evaluate LDA model
    
    Returns:
    --------
    metrics : dict
        Dictionary containing all performance metrics
    predictions : dict
        Model predictions and probabilities
    """
    # Feature scaling
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train LDA model
    lda = LinearDiscriminantAnalysis(solver='svd', store_covariance=True)
    
    # Cross-validation
    cv_scores = cross_val_score(lda, X_train_scaled, y_train, cv=5, scoring='recall')
    
    # Fit model
    lda.fit(X_train_scaled, y_train)
    y_pred = lda.predict(X_test_scaled)
    y_proba = lda.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    youden_index = recall + specificity - 1
    
    # Calculate ROC-AUC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    
    metrics = {
        'Model': model_name,
        'Num_Features': X_train.shape[1],
        'CV_Recall_Mean': cv_scores.mean(),
        'CV_Recall_Std': cv_scores.std(),
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Specificity': specificity,
        'Youden_Index': youden_index,
        'AUC': roc_auc,
        'TP': tp,
        'TN': tn,
        'FP': fp,
        'FN': fn
    }
    
    predictions = {
        'y_pred': y_pred,
        'y_proba': y_proba,
        'fpr': fpr,
        'tpr': tpr
    }
    
    return metrics, predictions

print('Model evaluation function defined successfully')

In [ ]:
# Define data balancing function
def balance_data_smote_under(X_train, y_train):
    """
    Balance training data using SMOTE + UnderSampling
    """
    smote = SMOTE(random_state=random_seed, k_neighbors=5)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    
    under_sampler = RandomUnderSampler(random_state=random_seed)
    X_train_balanced, y_train_balanced = under_sampler.fit_resample(X_train_smote, y_train_smote)
    
    return X_train_balanced, y_train_balanced

print('Data balancing function defined successfully')

In [ ]:
# ========== IG THRESHOLD EXPERIMENTS: ORIGINAL LDA ==========
print('\n' + '='*100)
print('IG THRESHOLD SELECTION - ORIGINAL LDA MODEL (Imbalanced Data)')
print('='*100)

# Define IG thresholds to test
ig_thresholds = [1.0, 0.7, 0.5, 0.2]
threshold_names = ['All (1.0)', 'Top 70%', 'Top 50%', 'Top 20%']

# Store results
results_original = []
predictions_original = {}

# Run experiments for each threshold
for threshold, name in zip(ig_thresholds, threshold_names):
    print(f'\n--- Testing IG Threshold: {name} (threshold={threshold}) ---')
    
    # Feature selection
    X_train_sel, X_test_sel, selected_features, scores = select_features_by_threshold(
        X_train_full, y_train_full, X_test_full, threshold
    )
    
    print(f'Features selected: {X_train_sel.shape[1]}')
    
    # Train and evaluate model
    metrics, predictions = evaluate_lda_model(
        X_train_sel, y_train_full, X_test_sel, y_test_full,
        model_name=f'Original_{name}'
    )
    
    results_original.append(metrics)
    predictions_original[name] = predictions
    
    # Print results
    print(f'Accuracy: {metrics["Accuracy"]:.4f}, Recall: {metrics["Recall"]:.4f}, AUC: {metrics["AUC"]:.4f}')

print('\nOriginal LDA experiments completed!')

In [ ]:
# ========== IG THRESHOLD EXPERIMENTS: SMOTE+UnderSampling LDA ==========
print('\n' + '='*100)
print('IG THRESHOLD SELECTION - SMOTE+UnderSampling LDA MODEL (Balanced Data)')
print('='*100)

# Store results
results_balanced = []
predictions_balanced = {}

# Run experiments for each threshold
for threshold, name in zip(ig_thresholds, threshold_names):
    print(f'\n--- Testing IG Threshold: {name} (threshold={threshold}) ---')
    
    # Feature selection
    X_train_sel, X_test_sel, selected_features, scores = select_features_by_threshold(
        X_train_full, y_train_full, X_test_full, threshold
    )
    
    print(f'Features selected: {X_train_sel.shape[1]}')
    
    # Data balancing
    X_train_balanced, y_train_balanced = balance_data_smote_under(X_train_sel, y_train_full)
    print(f'Balanced training data shape: {X_train_balanced.shape}')
    
    # Train and evaluate model
    metrics, predictions = evaluate_lda_model(
        X_train_balanced, y_train_balanced, X_test_sel, y_test_full,
        model_name=f'Balanced_{name}'
    )
    
    results_balanced.append(metrics)
    predictions_balanced[name] = predictions
    
    # Print results
    print(f'Accuracy: {metrics["Accuracy"]:.4f}, Recall: {metrics["Recall"]:.4f}, AUC: {metrics["AUC"]:.4f}')

print('\nSMOTE+UnderSampling LDA experiments completed!')

In [ ]:
# ========== COMPREHENSIVE COMPARISON TABLE ==========
print('\n' + '='*130)
print('COMPREHENSIVE RESULTS TABLE: IG THRESHOLD IMPACT')
print('='*130)

# Create comparison dataframe
comparison_data = []

for i, (orig_metrics, bal_metrics) in enumerate(zip(results_original, results_balanced)):
    threshold_name = threshold_names[i]
    
    comparison_data.append({
        'IG_Threshold': threshold_name,
        'Features': orig_metrics['Num_Features'],
        'Model_Type': 'Original',
        'Accuracy': orig_metrics['Accuracy'],
        'Precision': orig_metrics['Precision'],
        'Recall': orig_metrics['Recall'],
        'F1-Score': orig_metrics['F1-Score'],
        'Specificity': orig_metrics['Specificity'],
        'Youden_Index': orig_metrics['Youden_Index'],
        'AUC': orig_metrics['AUC']
    })
    
    comparison_data.append({
        'IG_Threshold': threshold_name,
        'Features': bal_metrics['Num_Features'],
        'Model_Type': 'SMOTE+Under',
        'Accuracy': bal_metrics['Accuracy'],
        'Precision': bal_metrics['Precision'],
        'Recall': bal_metrics['Recall'],
        'F1-Score': bal_metrics['F1-Score'],
        'Specificity': bal_metrics['Specificity'],
        'Youden_Index': bal_metrics['Youden_Index'],
        'AUC': bal_metrics['AUC']
    })

comparison_df = pd.DataFrame(comparison_data)

# Display with formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print(comparison_df.to_string(index=False))
print('='*130)

In [ ]:
# ========== COMPARISON BY IG THRESHOLD ==========
print('\n' + '='*100)
print('DETAILED COMPARISON: Original vs SMOTE+UnderSampling BY IG THRESHOLD')
print('='*100)

for i, (orig_metrics, bal_metrics, threshold_name) in enumerate(zip(results_original, results_balanced, threshold_names)):
    print(f'\n>>> IG Threshold: {threshold_name} ({ig_thresholds[i]}) [{orig_metrics["Num_Features"]} features]')
    print('-' * 100)
    
    # Compare metrics
    metrics_to_compare = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity', 'Youden_Index', 'AUC']
    
    for metric in metrics_to_compare:
        orig_val = orig_metrics[metric]
        bal_val = bal_metrics[metric]
        diff = bal_val - orig_val
        pct = (diff / abs(orig_val)) * 100 if orig_val != 0 else 0
        symbol = '↑' if diff > 0 else '↓' if diff < 0 else '='
        
        print(f'  {metric:15s}: Original {orig_val:.4f} → Balanced {bal_val:.4f} ({symbol} Δ: {diff:+.4f}, {pct:+.2f}%)')

print('\n' + '='*100)

In [ ]:
# ========== FIND OPTIMAL IG THRESHOLD ==========
print('\n' + '='*100)
print('OPTIMAL IG THRESHOLD ANALYSIS')
print('='*100)

# Find best threshold for each model type and metric
print('\n>>> Best IG Threshold for ORIGINAL LDA:')
orig_df = pd.DataFrame(results_original)

for metric in ['Accuracy', 'Recall', 'F1-Score', 'Youden_Index', 'AUC']:
    best_idx = orig_df[metric].idxmax()
    best_threshold = threshold_names[best_idx]
    best_value = orig_df.loc[best_idx, metric]
    best_features = orig_df.loc[best_idx, 'Num_Features']
    print(f'  {metric:15s}: {best_threshold:12s} ({best_features} features) = {best_value:.4f}')

print('\n>>> Best IG Threshold for SMOTE+UnderSampling LDA:')
bal_df = pd.DataFrame(results_balanced)

for metric in ['Accuracy', 'Recall', 'F1-Score', 'Youden_Index', 'AUC']:
    best_idx = bal_df[metric].idxmax()
    best_threshold = threshold_names[best_idx]
    best_value = bal_df.loc[best_idx, metric]
    best_features = bal_df.loc[best_idx, 'Num_Features']
    print(f'  {metric:15s}: {best_threshold:12s} ({best_features} features) = {best_value:.4f}')

print('\n' + '='*100)

In [ ]:
# ========== VISUALIZATION: IG THRESHOLD IMPACT ON ORIGINAL LDA ==========
orig_df = pd.DataFrame(results_original)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('IG Threshold Impact - Original LDA Model', fontsize=16, fontweight='bold')

# Accuracy vs IG Threshold
axes[0, 0].plot(threshold_names, orig_df['Accuracy'], marker='o', linewidth=2.5, markersize=8, color='steelblue')
axes[0, 0].set_title('Accuracy vs IG Threshold', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim([0, 1])

# Recall vs IG Threshold
axes[0, 1].plot(threshold_names, orig_df['Recall'], marker='s', linewidth=2.5, markersize=8, color='darkorange')
axes[0, 1].set_title('Recall vs IG Threshold', fontweight='bold')
axes[0, 1].set_ylabel('Recall')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 1])

# F1-Score vs IG Threshold
axes[1, 0].plot(threshold_names, orig_df['F1-Score'], marker='^', linewidth=2.5, markersize=8, color='green')
axes[1, 0].set_title('F1-Score vs IG Threshold', fontweight='bold')
axes[1, 0].set_ylabel('F1-Score')
axes[1, 0].set_xlabel('IG Threshold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1])

# AUC vs IG Threshold
axes[1, 1].plot(threshold_names, orig_df['AUC'], marker='d', linewidth=2.5, markersize=8, color='red')
axes[1, 1].set_title('AUC vs IG Threshold', fontweight='bold')
axes[1, 1].set_ylabel('AUC')
axes[1, 1].set_xlabel('IG Threshold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

In [ ]:
# ========== VISUALIZATION: IG THRESHOLD IMPACT ON SMOTE+UnderSampling LDA ==========
bal_df = pd.DataFrame(results_balanced)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('IG Threshold Impact - SMOTE+UnderSampling LDA Model', fontsize=16, fontweight='bold')

# Accuracy vs IG Threshold
axes[0, 0].plot(threshold_names, bal_df['Accuracy'], marker='o', linewidth=2.5, markersize=8, color='steelblue')
axes[0, 0].set_title('Accuracy vs IG Threshold', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim([0, 1])

# Recall vs IG Threshold
axes[0, 1].plot(threshold_names, bal_df['Recall'], marker='s', linewidth=2.5, markersize=8, color='darkorange')
axes[0, 1].set_title('Recall vs IG Threshold', fontweight='bold')
axes[0, 1].set_ylabel('Recall')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 1])

# F1-Score vs IG Threshold
axes[1, 0].plot(threshold_names, bal_df['F1-Score'], marker='^', linewidth=2.5, markersize=8, color='green')
axes[1, 0].set_title('F1-Score vs IG Threshold', fontweight='bold')
axes[1, 0].set_ylabel('F1-Score')
axes[1, 0].set_xlabel('IG Threshold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1])

# AUC vs IG Threshold
axes[1, 1].plot(threshold_names, bal_df['AUC'], marker='d', linewidth=2.5, markersize=8, color='red')
axes[1, 1].set_title('AUC vs IG Threshold', fontweight='bold')
axes[1, 1].set_ylabel('AUC')
axes[1, 1].set_xlabel('IG Threshold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

In [ ]:
# ========== COMPARATIVE VISUALIZATION: BOTH MODELS ==========
orig_df = pd.DataFrame(results_original)
bal_df = pd.DataFrame(results_balanced)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('IG Threshold Impact Comparison: Original vs SMOTE+UnderSampling', fontsize=16, fontweight='bold')

# Accuracy comparison
axes[0, 0].plot(threshold_names, orig_df['Accuracy'], marker='o', linewidth=2.5, markersize=8, label='Original', color='steelblue')
axes[0, 0].plot(threshold_names, bal_df['Accuracy'], marker='o', linewidth=2.5, markersize=8, label='SMOTE+Under', color='darkorange')
axes[0, 0].set_title('Accuracy vs IG Threshold', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend(loc='best')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim([0, 1])

# Recall comparison
axes[0, 1].plot(threshold_names, orig_df['Recall'], marker='s', linewidth=2.5, markersize=8, label='Original', color='steelblue')
axes[0, 1].plot(threshold_names, bal_df['Recall'], marker='s', linewidth=2.5, markersize=8, label='SMOTE+Under', color='darkorange')
axes[0, 1].set_title('Recall vs IG Threshold', fontweight='bold')
axes[0, 1].set_ylabel('Recall')
axes[0, 1].legend(loc='best')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 1])

# F1-Score comparison
axes[1, 0].plot(threshold_names, orig_df['F1-Score'], marker='^', linewidth=2.5, markersize=8, label='Original', color='steelblue')
axes[1, 0].plot(threshold_names, bal_df['F1-Score'], marker='^', linewidth=2.5, markersize=8, label='SMOTE+Under', color='darkorange')
axes[1, 0].set_title('F1-Score vs IG Threshold', fontweight='bold')
axes[1, 0].set_ylabel('F1-Score')
axes[1, 0].set_xlabel('IG Threshold')
axes[1, 0].legend(loc='best')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1])

# AUC comparison
axes[1, 1].plot(threshold_names, orig_df['AUC'], marker='d', linewidth=2.5, markersize=8, label='Original', color='steelblue')
axes[1, 1].plot(threshold_names, bal_df['AUC'], marker='d', linewidth=2.5, markersize=8, label='SMOTE+Under', color='darkorange')
axes[1, 1].set_title('AUC vs IG Threshold', fontweight='bold')
axes[1, 1].set_ylabel('AUC')
axes[1, 1].set_xlabel('IG Threshold')
axes[1, 1].legend(loc='best')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

In [ ]:
# ========== ROC CURVE COMPARISON: ALL IG THRESHOLDS ==========
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('ROC Curves by IG Threshold - Original vs SMOTE+UnderSampling LDA', fontsize=16, fontweight='bold')

for i, (threshold_name, ax) in enumerate(zip(threshold_names, axes.flat)):
    # Original ROC
    fpr_orig = predictions_original[threshold_name]['fpr']
    tpr_orig = predictions_original[threshold_name]['tpr']
    auc_orig = results_original[i]['AUC']
    
    # Balanced ROC
    fpr_bal = predictions_balanced[threshold_name]['fpr']
    tpr_bal = predictions_balanced[threshold_name]['tpr']
    auc_bal = results_balanced[i]['AUC']
    
    # Plot
    ax.plot(fpr_orig, tpr_orig, color='steelblue', lw=2.5, 
            label=f'Original (AUC={auc_orig:.4f})')
    ax.plot(fpr_bal, tpr_bal, color='darkorange', lw=2.5, 
            label=f'SMOTE+Under (AUC={auc_bal:.4f})')
    ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', alpha=0.5)
    
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=10)
    ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(f'IG Threshold: {threshold_name} ({results_original[i]["Num_Features"]} features)', fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ========== FINAL SUMMARY ==========
print('\n' + '='*100)
print('FINAL SUMMARY: IG THRESHOLD SELECTION IMPACT')
print('='*100)

orig_df = pd.DataFrame(results_original)
bal_df = pd.DataFrame(results_balanced)

print('\n>>> KEY FINDINGS:')
print('\n1. Feature Reduction Impact:')
print(f'   - All features (1.0): {results_original[0]["Num_Features"]} features')
print(f'   - Top 70% (0.7): {results_original[1]["Num_Features"]} features')
print(f'   - Top 50% (0.5): {results_original[2]["Num_Features"]} features')
print(f'   - Top 20% (0.2): {results_original[3]["Num_Features"]} features')

print('\n2. Original LDA Performance Range:')
print(f'   Accuracy:  {orig_df["Accuracy"].min():.4f} - {orig_df["Accuracy"].max():.4f} (Δ: {orig_df["Accuracy"].max() - orig_df["Accuracy"].min():.4f})')
print(f'   Recall:    {orig_df["Recall"].min():.4f} - {orig_df["Recall"].max():.4f} (Δ: {orig_df["Recall"].max() - orig_df["Recall"].min():.4f})')
print(f'   F1-Score:  {orig_df["F1-Score"].min():.4f} - {orig_df["F1-Score"].max():.4f} (Δ: {orig_df["F1-Score"].max() - orig_df["F1-Score"].min():.4f})')
print(f'   AUC:       {orig_df["AUC"].min():.4f} - {orig_df["AUC"].max():.4f} (Δ: {orig_df["AUC"].max() - orig_df["AUC"].min():.4f})')

print('\n3. SMOTE+UnderSampling LDA Performance Range:')
print(f'   Accuracy:  {bal_df["Accuracy"].min():.4f} - {bal_df["Accuracy"].max():.4f} (Δ: {bal_df["Accuracy"].max() - bal_df["Accuracy"].min():.4f})')
print(f'   Recall:    {bal_df["Recall"].min():.4f} - {bal_df["Recall"].max():.4f} (Δ: {bal_df["Recall"].max() - bal_df["Recall"].min():.4f})')
print(f'   F1-Score:  {bal_df["F1-Score"].min():.4f} - {bal_df["F1-Score"].max():.4f} (Δ: {bal_df["F1-Score"].max() - bal_df["F1-Score"].min():.4f})')
print(f'   AUC:       {bal_df["AUC"].min():.4f} - {bal_df["AUC"].max():.4f} (Δ: {bal_df["AUC"].max() - bal_df["AUC"].min():.4f})')

print('\n4. Recommended IG Threshold:')
# Find best balanced AUC
best_bal_idx = bal_df['AUC'].idxmax()
best_threshold_name = threshold_names[best_bal_idx]
best_features = bal_df.loc[best_bal_idx, 'Num_Features']
print(f'   Based on SMOTE+UnderSampling AUC: {best_threshold_name} ({best_features} features)')
print(f'   Achieved Recall: {bal_df.loc[best_bal_idx, "Recall"]:.4f}')
print(f'   Achieved AUC: {bal_df.loc[best_bal_idx, "AUC"]:.4f}')
print(f'   Achieved F1-Score: {bal_df.loc[best_bal_idx, "F1-Score"]:.4f}')

print('\n' + '='*100)